In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
df = spark.read.table('fintech360.data_quality_passed.loans')
df.limit(5).display()

In [0]:

from pyspark.sql import functions as F

# ============================================================
# 1. READ DATA QUALITY PASSED LOANS
# ============================================================

source_df = spark.read.table(
    "fintech360.data_quality_passed.loans"
)

# ============================================================
# 2. STANDARDIZE BUSINESS COLUMNS
# ============================================================

df = (source_df
    .withColumn("business_loan_status", F.upper(F.trim(F.col("loan_status"))))
    .withColumn("business_loan_source", F.upper(F.trim(F.col("loan_source"))))
    .withColumn("business_loan_purpose", F.upper(F.trim(F.col("loan_purpose"))))
    .withColumn("business_application_channel", F.upper(F.trim(F.col("application_channel"))))
    .withColumn("business_currency", F.upper(F.trim(F.col("currency"))))
    .withColumn("business_principal", F.col("principal_amount").cast("double"))
    .withColumn("business_annual_interest_rate_pct", F.col("annual_interest_rate_pct").cast("double"))
    .withColumn("business_tenure_months", F.col("tenure_months").cast("int"))
    .withColumn("_application_ts", F.col("application_ts").cast("long"))
    .withColumn("_disbursement_ts", F.col("disbursement_ts").cast("long"))
    .withColumn("_maturity_ts", F.col("maturity_date").cast("long"))
)

# ============================================================
# 3. CONVERT UNIX TIMESTAMPS TO DATETIME
# All three lifecycle timestamps are assumed to be Unix seconds.
# ============================================================

df = (df
    .withColumn(
        "business_application_datetime",
        F.to_timestamp(F.from_unixtime(F.col("_application_ts")))
    )
    .withColumn(
        "business_disbursement_datetime",
        F.to_timestamp(F.from_unixtime(F.col("_disbursement_ts")))
    )
    .withColumn(
        "business_maturity_datetime",
        F.to_timestamp(F.from_unixtime(F.col("_maturity_ts")))
    )
)

# Application date dimensions
df = (df
    .withColumn("business_application_date", F.to_date("business_application_datetime"))
    .withColumn("business_application_month", F.date_format("business_application_datetime", "yyyy-MM"))
    .withColumn("business_application_year", F.year("business_application_datetime"))
    .withColumn("business_application_day_of_week", F.date_format("business_application_datetime", "EEEE"))
    .withColumn("business_disbursement_date", F.to_date("business_disbursement_datetime"))
    .withColumn("business_disbursement_month", F.date_format("business_disbursement_datetime", "yyyy-MM"))
    .withColumn("business_maturity_date", F.to_date("business_maturity_datetime"))
)

# ============================================================
# 4. LOAN LIFECYCLE FLAGS
# These flags classify the current status, not historical events.
# ============================================================

df = (df
    .withColumn(
        "application_flag",
        F.when(
            F.col("business_loan_status").isin(
                "APPLIED", "APPROVED", "DISBURSED",
                "REJECTED", "CLOSED"
            ), 1
        ).otherwise(0)
    )
    .withColumn(
        "approved_flag",
        F.when(
            F.col("business_loan_status").isin(
                "APPROVED", "DISBURSED", "CLOSED"
            ), 1
        ).otherwise(0)
    )
    .withColumn(
        "disbursed_flag",
        F.when(
            F.col("_disbursement_ts").isNotNull() &
            F.col("business_loan_status").isin("DISBURSED", "CLOSED"),
            1
        ).otherwise(0)
    )
    .withColumn(
        "rejected_flag",
        F.when(F.col("business_loan_status") == "REJECTED", 1)
         .otherwise(0)
    )
    .withColumn(
        "closed_flag",
        F.when(F.col("business_loan_status") == "CLOSED", 1)
         .otherwise(0)
    )
    .withColumn(
        "approved_not_disbursed_flag",
        F.when(
            (F.col("business_loan_status") == "APPROVED") &
            F.col("_disbursement_ts").isNull(),
            1
        ).otherwise(0)
    )
)

# ============================================================
# 5. DISBURSED PRINCIPAL AND PIPELINE AMOUNTS
# ============================================================

df = (df
    .withColumn(
        "disbursed_principal_amount",
        F.when(
            F.col("disbursed_flag") == 1,
            F.col("business_principal")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "approved_pipeline_amount",
        F.when(
            F.col("approved_not_disbursed_flag") == 1,
            F.col("business_principal")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "non_disbursed_application_amount",
        F.when(
            (F.col("disbursed_flag") == 0) &
            (F.col("business_loan_status") == "APPLIED"),
            F.col("business_principal")
        ).otherwise(F.lit(0.0))
    )
)

# ============================================================
# 6. APPLICATION-TO-DISBURSAL TURNAROUND
# ============================================================

df = (df
    .withColumn(
        "application_to_disbursal_hours",
        F.when(
            (F.col("_application_ts").isNotNull()) &
            (F.col("_disbursement_ts").isNotNull()) &
            (F.col("_disbursement_ts") >= F.col("_application_ts")),
            F.round(
                (F.col("_disbursement_ts") - F.col("_application_ts")) / 3600.0,
                2
            )
        )
    )
    .withColumn(
        "application_to_disbursal_days",
        F.when(
            F.col("application_to_disbursal_hours").isNotNull(),
            F.round(F.col("application_to_disbursal_hours") / 24.0, 2)
        )
    )
    .withColumn(
        "disbursal_tat_bucket",
        F.when(
            F.col("disbursed_flag") == 0, "NOT_DISBURSED"
        ).when(
            F.col("application_to_disbursal_hours").isNull(),
            "INVALID_OR_MISSING_TIMESTAMPS"
        ).when(
            F.col("application_to_disbursal_hours") <= 24,
            "WITHIN_24_HOURS"
        ).when(
            F.col("application_to_disbursal_hours") <= 72,
            "1_TO_3_DAYS"
        ).when(
            F.col("application_to_disbursal_hours") <= 168,
            "4_TO_7_DAYS"
        ).otherwise("MORE_THAN_7_DAYS")
    )
)

# ============================================================
# 7. LOAN PRICING FEATURES
# Estimated simple interest, not the contractual repayment amount.
# Excludes fees, compounding, prepayments and amortization effects.
# ============================================================

df = (df
    .withColumn(
        "estimated_simple_interest_amount",
        F.when(
            (F.col("business_principal") > 0) &
            (F.col("business_annual_interest_rate_pct") >= 0) &
            (F.col("business_tenure_months") > 0),
            F.round(
                F.col("business_principal") *
                F.col("business_annual_interest_rate_pct") / 100 *
                F.col("business_tenure_months") / 12,
                2
            )
        )
    )
    .withColumn(
        "estimated_principal_plus_simple_interest",
        F.when(
            F.col("estimated_simple_interest_amount").isNotNull(),
            F.col("business_principal") +
            F.col("estimated_simple_interest_amount")
        )
    )
    .withColumn(
        "monthly_nominal_interest_rate_pct",
        F.round(
            F.col("business_annual_interest_rate_pct") / 12,
            4
        )
    )
)

# ============================================================
# 8. PRINCIPAL AMOUNT BANDS
# ============================================================

df = df.withColumn(
    "principal_amount_band",
    F.when(F.col("business_principal") < 5000, "01_BELOW_5000")
     .when(F.col("business_principal") < 10000, "02_5000_TO_9999")
     .when(F.col("business_principal") < 20000, "03_10000_TO_19999")
     .when(F.col("business_principal") < 50000, "04_20000_TO_49999")
     .otherwise("05_50000_PLUS")
)

# ============================================================
# 9. TENURE SEGMENTATION
# ============================================================

df = df.withColumn(
    "tenure_band",
    F.when(F.col("business_tenure_months") <= 3, "SHORT_TENURE")
     .when(F.col("business_tenure_months") <= 6, "MEDIUM_SHORT_TENURE")
     .when(F.col("business_tenure_months") <= 12, "MEDIUM_TENURE")
     .when(F.col("business_tenure_months") <= 24, "LONG_TENURE")
     .otherwise("VERY_LONG_TENURE")
)

# ============================================================
# 10. INTEREST RATE SEGMENTATION
# Thresholds are configurable analytical bands, not risk grades.
# ============================================================

df = df.withColumn(
    "interest_rate_band",
    F.when(F.col("business_annual_interest_rate_pct") < 12, "BELOW_12_PCT")
     .when(F.col("business_annual_interest_rate_pct") < 24, "12_TO_24_PCT")
     .when(F.col("business_annual_interest_rate_pct") < 36, "24_TO_36_PCT")
     .otherwise("36_PCT_PLUS")
)

# ============================================================
# 11. SOURCE AND CHANNEL SEGMENTATION
# ============================================================

df = (df
    .withColumn(
        "acquisition_source_group",
        F.when(
            F.col("business_loan_source").isin("REFERRAL", "PARTNER"),
            "PARTNER_AND_REFERRAL"
        ).when(
            F.col("business_loan_source") == "WEB",
            "DIGITAL_WEB"
        ).when(
            F.col("business_loan_source").isNull(),
            "UNKNOWN_SOURCE"
        ).otherwise("OTHER_SOURCE")
    )
    .withColumn(
        "application_channel_group",
        F.when(
            F.col("business_application_channel").isin("WEB", "APP"),
            "DIGITAL"
        ).when(
            F.col("business_application_channel").isin("PARTNER", "REFERRAL"),
            "PARTNER_AND_REFERRAL"
        ).when(
            F.col("business_application_channel").isNull(),
            "UNKNOWN_CHANNEL"
        ).otherwise("OTHER_CHANNEL")
    )
)

# ============================================================
# 12. MATURITY AND REVIEW SIGNALS
# ============================================================

df = (df
    .withColumn(
        "maturity_date_missing_flag",
        F.when(
            (F.col("disbursed_flag") == 1) &
            F.col("_maturity_ts").isNull(),
            1
        ).otherwise(0)
    )
    .withColumn(
        "maturity_before_disbursal_flag",
        F.when(
            F.col("_maturity_ts").isNotNull() &
            F.col("_disbursement_ts").isNotNull() &
            (F.col("_maturity_ts") < F.col("_disbursement_ts")),
            1
        ).otherwise(0)
    )
    .withColumn(
        "high_interest_rate_review_flag",
        F.when(
            F.col("business_annual_interest_rate_pct") > 36,
            1
        ).otherwise(0)
    )
    .withColumn(
        "loan_review_signal",
        F.when(
            F.col("approved_not_disbursed_flag") == 1,
            "APPROVED_PENDING_DISBURSAL"
        ).when(
            F.col("maturity_before_disbursal_flag") == 1,
            "MATURITY_BEFORE_DISBURSAL"
        ).when(
            F.col("maturity_date_missing_flag") == 1,
            "MISSING_MATURITY_DATE"
        ).when(
            F.col("high_interest_rate_review_flag") == 1,
            "HIGH_INTEREST_RATE_REVIEW"
        ).otherwise("NO_BASIC_REVIEW_SIGNAL")
    )
)

# ============================================================
# 13. TRANSFORMATION METADATA
# ============================================================

df = df.withColumn(
    "business_transformed_at",
    F.current_timestamp()
)

# ============================================================
# 14. FINAL BUSINESS DATAFRAME
# ============================================================

loans_business = df

display(loans_business.limit(10))


In [0]:

# Create destination schema
spark.sql("""
CREATE SCHEMA IF NOT EXISTS fintech360.business_transformation
""")

# Append transformed loan records
(loans_business.write
    .format("delta")
    .mode("append")
    .saveAsTable("fintech360.business_transformation.loans"))

print("Loan business transformation completed.")
print("Target: fintech360.business_transformation.loans")
